In [1]:
import sys
import time
import torch
import torchvision
from pathlib import Path
import os
import numpy as np
from torch.utils.data import Dataset
from PIL import Image
from visdrone_toolkit import VisDroneDataset
from visdrone_toolkit.utils import collate_fn
from torchvision.models.detection import (
    retinanet_resnet50_fpn,
    RetinaNet_ResNet50_FPN_Weights
)
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from torchmetrics.detection import MeanAveragePrecision


C:\Users\Ray\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
train_img = r"D:\cv\Dataset\VisDrone2019-DET-train\images"
train_annotations =r"D:\cv\Dataset\VisDrone2019-DET-train\annotations"

val_img = r"D:\cv\Dataset\VisDrone2019-DET-val\images"
val_annotations= r"D:\cv\Dataset\VisDrone2019-DET-val\annotations"

test_img = r"D:\cv\Dataset\VisDrone2019-DET-test-dev\images"
test_annotations = r"D:\cv\Dataset\VisDrone2019-DET-test-dev\annotations"





In [3]:
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

torch.set_float32_matmul_precision("high")

weights = RetinaNet_ResNet50_FPN_Weights.DEFAULT

model = retinanet_resnet50_fpn(
    weights=weights,
)

device = torch.device("cuda")
model = model.to(device)

In [4]:
train_dataset = VisDroneDataset(
    image_dir=train_img,
    annotation_dir=train_annotations,
    filter_ignored=True,
    filter_crowd=True,
)



Found 6471 images in D:\cv\Dataset\VisDrone2019-DET-train\images


In [5]:
image, target = train_dataset[0]

print("image:", image.shape)

for key, value in target.items():
    print(
        key,
        tuple(value.shape) if torch.is_tensor(value) else type(value),
        value.dtype if torch.is_tensor(value) else ""
    )

image: torch.Size([3, 450, 800])
boxes (82, 4) torch.float32
labels (82,) torch.int64
image_id (1,) torch.int64
area (82,) torch.float32
iscrowd (82,) torch.int64


In [6]:
train_loader = DataLoader(
    train_dataset,
    batch_size=8,
    shuffle=True,
    num_workers=8,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=4,
    collate_fn=collate_fn,
)


model.transform.min_size = (800,)
model.transform.max_size = 800


optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.0025,
    momentum=0.9,
    weight_decay=0.0005
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=450
)

scaler = torch.amp.GradScaler("cuda")

EPOCHS = 100

In [8]:
for epoch in range(EPOCHS):

    model.train()
    epoch_loss = 0.0

    progress_bar = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}",
        unit="batch",
        dynamic_ncols=True
    )

    for images, targets in progress_bar:

        images = [
            image.to(device, non_blocking=True)
            for image in images
        ]

        targets = [
            {
                key: value.to(device, non_blocking=True)
                if torch.is_tensor(value) else value
                for key, value in target.items()
            }
            for target in targets
        ]

        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            loss_dict = model(images, targets)
            loss = sum(loss_dict.values())

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}",
            cls=f"{loss_dict['classification'].item():.3f}",
            box=f"{loss_dict['bbox_regression'].item():.3f}")

    epoch_loss /= len(train_loader)

    scheduler.step()

    print(
        f"Epoch [{epoch + 1}/{EPOCHS}] "
        f"Train Loss: {epoch_loss:.4f} | "
        f"LR: {optimizer.param_groups[0]['lr']:.7f}"
    )

print("\nTraining finished.")

torch.save(
    model.state_dict(),
    "retinanet_visdrone_100ep.pth"
)

print("Model saved: retinanet_visdrone_100ep.pth")

Epoch 1/100: 100%|██████████| 809/809 [04:21<00:00,  3.09batch/s, box=0.507, cls=0.306, loss=0.8133]  


Epoch [1/100] Train Loss: 0.9251 | LR: 0.0025000


Epoch 2/100: 100%|██████████| 809/809 [03:17<00:00,  4.11batch/s, box=0.515, cls=0.284, loss=0.7987]


Epoch [2/100] Train Loss: 0.8172 | LR: 0.0024999


Epoch 3/100: 100%|██████████| 809/809 [03:17<00:00,  4.10batch/s, box=0.368, cls=0.212, loss=0.5804]


...

Epoch [98/100] Train Loss: 0.2816 | LR: 0.0022187


Epoch 99/100: 100%|██████████| 809/809 [03:30<00:00,  3.85batch/s, box=0.166, cls=0.066, loss=0.2317]


Epoch [99/100] Train Loss: 0.2995 | LR: 0.0022131


Epoch 100/100: 100%|██████████| 809/809 [03:30<00:00,  3.83batch/s, box=0.152, cls=0.052, loss=0.2034]

Epoch [100/100] Train Loss: 0.2788 | LR: 0.0022076

Training finished.
Model saved: retinanet_visdrone_100ep.pth
